In [1]:
import pandas as pd 
master = pd.read_parquet("D:\\instacart_market_analysis\\master_table.parquet")


day_map = {0:"Sunday", 1:"Monday", 2:"Tuesday", 3:"Wednesday",
           4:"Thursday", 5:"Friday", 6:"Saturday"}

In [2]:
# order by hour
orders_by_hour = (master.groupby("order_hour_of_day")["order_id"]
                 .nunique()
                 .reset_index()
                 .rename(columns={"order_id":"total_orders"}))             
print(orders_by_hour)

    order_hour_of_day  total_orders
0                   0         21372
1                   1         11596
2                   2          7070
3                   3          5120
4                   4          5175
5                   5          8972
6                   6         28792
7                   7         86656
8                   8        168321
9                   9        243496
10                 10        271885
11                 11        268006
12                 12        256206
13                 13        261174
14                 14        265556
15                 15        266132
16                 16        255949
17                 17        214080
18                 18        170998
19                 19        131620
20                 20         98109
21                 21         73436
22                 22         57540
23                 23         37613


In [3]:
# orders by day of week
orders_by_dow = (master.groupby("order_dow")["order_id"]
                 .nunique()
                 .reset_index()
                 .rename(columns={"order_id": "total_orders"}))


orders_by_dow["Day_Name"] = orders_by_dow["order_dow"].map(day_map)
orders_by_dow = orders_by_dow[["Day_Name","order_dow","total_orders"]]
print(orders_by_dow)

    Day_Name  order_dow  total_orders
0     Sunday          0        557772
1     Monday          1        556705
2    Tuesday          2        441955
3  Wednesday          3        412400
4   Thursday          4        401212
5     Friday          5        425982
6   Saturday          6        418848


In [4]:
# orders by time of day bucket
orders_by_time = (master.groupby("time_of_order")["order_id"]
                  .nunique()
                  .reset_index()
                  .rename(columns={"order_id": "total_orders"})
                  .sort_values("total_orders", ascending=False))

print(orders_by_time)

  time_of_order  total_orders
0     Afternoon       1305017
2       Morning       1067156
1       Evening        614807
3         Night        227894


In [5]:
# orders by day type — weekend vs weekday
orders_by_daytype = (master.groupby("day_type")["order_id"]
                     .nunique()
                     .reset_index()
                     .rename(columns={"order_id": "total_orders"}))

print(orders_by_daytype)

  day_type  total_orders
0  Weekday       2238254
1  Weekend        976620


In [6]:
# average basket size by day of week
basket_by_dow = (master.groupby("order_dow")["basket_size"]
                 .mean()
                 .round(2)
                 .reset_index()
                 .rename(columns={"basket_size": "avg_basket_size"}))

basket_by_dow["day_name"] = basket_by_dow["order_dow"].map(day_map)
print(basket_by_dow)

   order_dow  avg_basket_size   day_name
0          0            16.48     Sunday
1          1            15.86     Monday
2          2            15.15    Tuesday
3          3            14.83  Wednesday
4          4            15.05   Thursday
5          5            15.65     Friday
6          6            16.29   Saturday


In [7]:
# average basket size by time of day
basket_by_time = (master.groupby("time_of_order")["basket_size"]
                  .mean()
                  .round(2)
                  .reset_index()
                  .rename(columns={"basket_size": "avg_basket_size"})
                  .sort_values("avg_basket_size", ascending=False))

print(basket_by_time)

  time_of_order  avg_basket_size
3         Night            16.73
2       Morning            16.06
0     Afternoon            15.61
1       Evening            14.84


In [8]:
# distribution of basket sizes
basket_dist = (master.groupby("order_id")["basket_size"]
               .first()
               .reset_index()
               .groupby("basket_size")["order_id"]
               .count()
               .reset_index()
               .rename(columns={"order_id": "order_count"})
               .sort_values("basket_size"))

print(basket_dist.head(20))

    basket_size  order_count
0             1       156748
1             2       186993
2             3       207027
3             4       222081
4             5       228330
5             6       227675
6             7       220006
7             8       203374
8             9       184347
9            10       165550
10           11       147461
11           12       131580
12           13       116871
13           14       103683
14           15        91644
15           16        81192
16           17        71360
17           18        62629
18           19        54817
19           20        48096


In [9]:
# distribution of days since prior order — exclude first orders
days_dist = (master[master["days_since_prior_order"] != -1]
             .groupby("days_since_prior_order")["order_id"]
             .nunique()
             .reset_index()
             .rename(columns={"order_id": "order_count"})
             .sort_values("days_since_prior_order"))

print(days_dist)

    days_since_prior_order  order_count
0                      0.0        64436
1                      1.0       141011
2                      2.0       187723
3                      3.0       210665
4                      4.0       214488
5                      5.0       206691
6                      6.0       230245
7                      7.0       306181
8                      8.0       173259
9                      9.0       112184
10                    10.0        90198
11                    11.0        76394
12                    12.0        71356
13                    13.0        77765
14                    14.0        93064
15                    15.0        61883
16                    16.0        43423
17                    17.0        36281
18                    18.0        33050
19                    19.0        31408
20                    20.0        35173
21                    21.0        41262
22                    22.0        29125
23                    23.0        21629


In [10]:
over_all_reorder_rate = master["reordered"].mean().round(4)
print(f"over all reorder rate is : {over_all_reorder_rate}")
print("As percentage",round(over_all_reorder_rate *100,2),"%")

over all reorder rate is : 0.5897
As percentage 58.97 %


In [11]:
reorder_by_dow = (master.groupby("order_dow")["reordered"]
                  .mean()
                  .round(4)
                  .reset_index()
                  .rename(columns={"reordered": "reorder_rate"}))

reorder_by_dow["day_name"] = reorder_by_dow["order_dow"].map(day_map)
reorder_by_dow["reorder_rate_pct"] = (reorder_by_dow["reorder_rate"] * 100).round(2)

reorder_by_dow = reorder_by_dow[[ "order_dow","day_name", "reorder_rate", "reorder_rate_pct"]]

print(reorder_by_dow)

   order_dow   day_name  reorder_rate  reorder_rate_pct
0          0     Sunday        0.5853             58.53
1          1     Monday        0.6038             60.38
2          2    Tuesday        0.5898             58.98
3          3  Wednesday        0.5863             58.63
4          4   Thursday        0.5910             59.10
5          5     Friday        0.5955             59.55
6          6   Saturday        0.5744             57.44


In [12]:
# reorder rate by time of day
reorder_by_time = (master.groupby("time_of_order")["reordered"]
                   .mean()
                   .round(4)
                   .reset_index()
                   .rename(columns={"reordered": "reorder_rate"}))

reorder_by_time["reorder_rate_pct"] = (reorder_by_time["reorder_rate"] * 100).round(2)
reorder_by_time = reorder_by_time.sort_values("reorder_rate", ascending=False)
print(reorder_by_time)

  time_of_order  reorder_rate  reorder_rate_pct
2       Morning        0.6110             61.10
3         Night        0.5821             58.21
0     Afternoon        0.5800             58.00
1       Evening        0.5755             57.55


In [13]:
# reorder rate by order frequency bucket
# do weekly shoppers reorder more than monthly shoppers
reorder_by_freq = (master.groupby("order_frequency")["reordered"]
                   .mean()
                   .round(4)
                   .reset_index()
                   .rename(columns={"reordered": "reorder_rate"}))

reorder_by_freq["reorder_rate_pct"] = (reorder_by_freq["reorder_rate"] * 100).round(2)
reorder_by_freq = reorder_by_freq.sort_values("reorder_rate", ascending=False)
print(reorder_by_freq)

  order_frequency  reorder_rate  reorder_rate_pct
3          Weekly        0.6745             67.45
0       Bi Weekly        0.6417             64.17
2         Monthly        0.5237             52.37
1     First order        0.0000              0.00


In [14]:
# reorder rate by order stage
# do loyal customers reorder more than new customers
reorder_by_stage = (master.groupby("order_stage")["reordered"]
                    .mean()
                    .round(4)
                    .reset_index()
                    .rename(columns={"reordered": "reorder_rate"}))

reorder_by_stage["reorder_rate_pct"] = (reorder_by_stage["reorder_rate"] * 100).round(2)
reorder_by_stage = reorder_by_stage.sort_values("reorder_rate", ascending=False)
print(reorder_by_stage)

    order_stage  reorder_rate  reorder_rate_pct
1         Loyal        0.7355             73.55
0       Growing        0.4991             49.91
2  New Customer        0.1352             13.52


In [15]:
# reorder rate by day type — weekend vs weekday
reorder_by_daytype = (master.groupby("day_type")["reordered"]
                      .mean()
                      .round(4)
                      .reset_index()
                      .rename(columns={"reordered": "reorder_rate"}))

reorder_by_daytype["reorder_rate_pct"] = (reorder_by_daytype["reorder_rate"] * 100).round(2)
print(reorder_by_daytype)

  day_type  reorder_rate  reorder_rate_pct
0  Weekday        0.5941             59.41
1  Weekend        0.5807             58.07


In [16]:
# one combined dow table
dow_combined = orders_by_dow.merge(basket_by_dow[["order_dow","avg_basket_size"]], on="order_dow", how="left")
dow_combined = dow_combined.merge(reorder_by_dow[["order_dow","reorder_rate","reorder_rate_pct"]], on="order_dow", how="left")
dow_combined.to_csv("D:\\instacart_market_analysis\\dow_analysis.csv", index=False)

In [17]:
# one combined time of day table
time_combined = orders_by_time.merge(basket_by_time[["time_of_order","avg_basket_size"]], on="time_of_order", how="left")
time_combined = time_combined.merge(reorder_by_time[["time_of_order","reorder_rate","reorder_rate_pct"]], on="time_of_order", how="left")
time_combined.to_csv("D:\\instacart_market_analysis\\time_analysis.csv", index=False)